# Prairie Vigie – notebook 2 : années 2024-2025 et vérification de la prédiction
À lancer **dans le même dossier que le notebook 1** (même Drive) : le cache 2015-2023 et les fichiers `sorties_prairies/` sont réutilisés.

### Règle de vérification (écrite avant les résultats)
- **Mesure 2025** : prairies permanentes 2025 de la région à moins de 10 % du chiffre officiel (614 875 ha, arrêté ratios 2025).
- **Prédiction** : le risque calculé avec les seules données 2023 (modèle de la v3, entraîné sur 2019-2021) est confrontée aux conversions
  **réellement observées entre 2024 et 2025**. Réussite si, sur la Normandie, les 10 % de prairies jugées les plus à risque
  contiennent **au moins 2 fois** plus de conversions que la moyenne (lift ≥ 2).
- Information : la campagne 2023→2024 est marquée par le régime d'interdiction de retournement, annoncé fin octobre 2023 puis levé rétroactivement au 1er janvier 2024 (arrêté du 25 novembre 2024) ;
  période atypique, analysée à part (notebook 3, bloc D).

In [ ]:
DOSSIER = "datathon_prairies"
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")                                   # point de montage standard de Colab
    os.makedirs(os.path.join("/content/drive/MyDrive", DOSSIER), exist_ok=True)
    os.chdir(os.path.join("/content/drive/MyDrive", DOSSIER))
except ImportError:
    pass                                                            # hors Colab : dossier courant
print("Dossier de travail :", os.getcwd())
if not os.path.exists("sorties_prairies"):
    print("⚠ Dossier vide : lancer d'abord les notebooks précédents dans ce même dossier.")

In [ ]:
# 0 bis. Bibliothèques
!pip -q install geopandas pyogrio pyarrow scikit-learn

In [ ]:
# 1. Paramètres
import json, time, gc, xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, requests
from shapely.geometry import shape, box
from shapely.validation import make_valid
from shapely.ops import unary_union

DEPS = ["14", "27", "50", "61", "76"]
HISTO = [2015, 2016, 2017, 2018]            # sert seulement à savoir si une parcelle a été cultivée
ANNEES_PP = [2019, 2020, 2021, 2022, 2023, 2024, 2025]   # définition « prairie permanente » stable à partir de 2019
ANNEES = HISTO + ANNEES_PP
TRANSITIONS = [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024), (2024, 2025)]
TAMPON = 5000                                # 5 km autour du département pour les variables de voisinage
WFS = "https://data.geopf.fr/wfs/ows"
PAGE, DALLE = 5000, 20_000
CACHE = Path("cache_rpg"); CACHE.mkdir(exist_ok=True)
SORTIES = Path("sorties_prairies"); SORTIES.mkdir(exist_ok=True)
SESSION = requests.Session()
CHAMPS = {"code_cultu", "code_group"}       # liste blanche : aucun identifiant d'exploitant

PP, HERBE, GEL = 18, {17, 18, 19}, {11, 12, 13}
ARABLE = {1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 24, 25, 26}
def classe(g):
    g = int(g) if pd.notna(g) else -1
    return ("pp" if g == PP else "herbe" if g in HERBE else "arable" if g in ARABLE
            else "gel" if g in GEL else "autre")

def polygone(g):
    """Géométrie valide, parties surfaciques seulement."""
    g = make_valid(g)
    if g.geom_type == "GeometryCollection":
        parts = [p for p in g.geoms if p.geom_type in ("Polygon", "MultiPolygon")]
        g = unary_union(parts) if parts else None
    return g
OFFICIEL_2025 = 614_875                      # ha de prairies permanentes 2025 (arrêté ratios 2025)

In [ ]:
# 2. Communes et couches RPG (2025 : couche « grandes catégories »)
def communes_dep(dep):
    f = CACHE / f"communes_{dep}.parquet"
    if f.exists():
        return gpd.read_parquet(f)
    r = SESSION.get(f"https://geo.api.gouv.fr/departements/{dep}/communes",
                    params={"geometry": "contour", "format": "geojson", "fields": "code,nom,codeEpci"}, timeout=180)
    r.raise_for_status()
    g = gpd.GeoDataFrame.from_features(r.json()["features"], crs=4326).to_crs(2154)
    g.to_parquet(f)
    return g

COMMUNES = gpd.GeoDataFrame(pd.concat([communes_dep(d).assign(dep=d) for d in DEPS], ignore_index=True),
                            geometry="geometry", crs=2154)
if "codeEpci" not in COMMUNES:
    COMMUNES["codeEpci"] = None
CONTOURS = {d: make_valid(unary_union(COMMUNES.loc[COMMUNES.dep == d, "geometry"].values)) for d in DEPS}
for d in DEPS:
    print(f"Département {d} : {(COMMUNES.dep == d).sum()} communes, {CONTOURS[d].area / 1e4:,.0f} ha")

r = SESSION.get(WFS, params={"SERVICE": "WFS", "REQUEST": "GetCapabilities", "VERSION": "2.0.0"}, timeout=180)
r.raise_for_status()
NOMS = {el.text for el in ET.fromstring(r.content).iter() if el.tag.endswith("}Name") and el.text}
COUCHES = {a: f"RPG.{a}:parcelles_graphiques" for a in ANNEES if f"RPG.{a}:parcelles_graphiques" in NOMS}
C2025 = "IGNF_RPG_GRANDES-CATEGORIES-PARCELLES_2025:grandes_categories_parcelles"
if 2025 not in COUCHES and C2025 in NOMS:
    COUCHES[2025] = C2025
print("Couches utilisées :", json.dumps(COUCHES, indent=1))
ANNEES = [a for a in ANNEES if a in COUCHES]
HISTO = [a for a in HISTO if a in COUCHES]



In [ ]:
# 3. Téléchargement 2024-2025 par dalles ADAPTATIVES : une dalle trop pleine est coupée en 4,
#    donc jamais de pagination (aucun risque de sauter des parcelles), contrôle du nombre annoncé.
def get(params):
    for essai in range(6):
        try:
            r = SESSION.get(WFS, params=params, timeout=300)
            if r.status_code == 200:
                return r.json()
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5 * 2 ** essai)
    raise RuntimeError(f"WFS indisponible : {params.get('TYPENAMES')}")

def dalle(couche, x0, y0, x1, y1, zone, sortie, stats, champs_bruts, vus):
    if not box(x0, y0, x1, y1).intersects(zone):
        return
    page = get({"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature", "TYPENAMES": couche,
                "SRSNAME": "EPSG:2154", "BBOX": f"{x0},{y0},{x1},{y1},EPSG:2154",
                "OUTPUTFORMAT": "application/json", "COUNT": PAGE})
    feats = page.get("features", [])
    m = page.get("numberMatched", page.get("totalFeatures"))
    if len(feats) >= PAGE or (isinstance(m, int) and m > len(feats)):
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2           # trop d'objets : on coupe en 4
        for a, b, c, d in [(x0, y0, xm, ym), (xm, y0, x1, ym), (x0, ym, xm, y1), (xm, ym, x1, y1)]:
            dalle(couche, a, b, c, d, zone, sortie, stats, champs_bruts, vus)
        return
    stats["annonces"] += m if isinstance(m, int) else len(feats)
    stats["recus"] += len(feats)
    for f in feats:
        cle = f.get("id") or json.dumps(f.get("geometry"))[:200]
        if cle in vus or not f.get("geometry"):
            continue
        vus.add(cle)
        props = {k.lower(): v for k, v in (f.get("properties") or {}).items()}
        champs_bruts |= set(props)
        sortie.append({**{k: props.get(k) for k in CHAMPS}, "geometry": shape(f["geometry"])})

def charger(dep, annee):
    fichier = CACHE / f"rpg_{dep}_{annee}_v3.parquet"
    if fichier.exists():
        return gpd.read_parquet(fichier)
    zone = CONTOURS[dep].buffer(TAMPON)
    xmin, ymin, xmax, ymax = zone.bounds
    lignes, stats, champs_bruts, vus = [], {"annonces": 0, "recus": 0}, set(), set()
    for x in np.arange(xmin, xmax, DALLE):
        for y in np.arange(ymin, ymax, DALLE):
            dalle(COUCHES[annee], x, y, x + DALLE, y + DALLE, zone, lignes, stats, champs_bruts, vus)
    (CACHE / f"complet_{dep}_{annee}.json").write_text(json.dumps(stats))
    manque = 1 - stats["recus"] / stats["annonces"] if stats["annonces"] else float("nan")
    print(f"    {dep} {annee} : {len(champs_bruts)} champs dans la source, {len(CHAMPS)} conservés ; manquants = {manque:.2%}")
    g = gpd.GeoDataFrame(lignes, geometry="geometry", crs=2154)
    g["geometry"] = g.geometry.apply(polygone)
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    g = g[g.representative_point().within(zone)].copy()
    g["dans_dep"] = g.representative_point().within(CONTOURS[dep])
    g["code_group"] = pd.to_numeric(g["code_group"], errors="coerce")
    g = g.reset_index(drop=True)
    g.to_parquet(fichier)
    return g

for d in DEPS:
    for a in ANNEES:
        t0 = time.time()
        n = len(charger(d, a))          # 2015-2023 : lus dans le cache de la v3
        if a >= 2024:
            print(f"{d} {a} : {n:,} parcelles ({time.time() - t0:.0f} s)")

In [ ]:
# 4. Contrôles 2024-2025 : nomenclature, surfaces, chiffre officiel 2025
g = charger("50", 2025)
print("Codes de culture du groupe 18 (prairies permanentes) en 2025 :",
      g.loc[g.code_group == 18, "code_cultu"].value_counts().head(5).to_dict())
lignes = []
for d in DEPS:
    for a in ANNEES:
        g = charger(d, a)
        g = g[g["dans_dep"]]
        s = (g.area / 1e4).groupby(g["code_group"].apply(classe)).sum()
        lignes.append({"dep": d, "annee": a, "n_parcelles": len(g), **s.round(0).to_dict()})
SURF = pd.DataFrame(lignes).fillna(0)
SURF.to_csv(SORTIES / "surfaces_par_dep_annee_2015_2025.csv", index=False)
reg = SURF.groupby("annee")[[c for c in ["n_parcelles", "pp", "herbe", "arable", "gel", "autre"] if c in SURF]].sum()
reg["variation_pp"] = reg["pp"].pct_change().round(3)
print(reg.to_string())
ecart = reg.loc[2025, "pp"] / OFFICIEL_2025 - 1
CRIT_2025 = bool(abs(ecart) <= 0.10)
print(f"\nPP 2025 RPG = {reg.loc[2025, 'pp']:,.0f} ha ; officiel = {OFFICIEL_2025:,} ha ; écart {ecart:+.1%} ->",
      "OK" if CRIT_2025 else "ÉCHEC")

In [ ]:
# 5. Fonctions : devenir des prairies, créations, variables, étiquettes à 2 ans (identiques au notebook 1)
def devenir(D, a0, a1):
    g0 = D[a0]
    pp = g0.loc[(g0["classe"] == "pp") & g0["dans_dep"], ["pid", "geometry"]].copy()
    pp["surf0"] = pp.area
    inter = gpd.overlay(pp[["pid", "geometry"]], D[a1][["classe", "geometry"]], how="intersection", keep_geom_type=True)
    inter["a"] = inter.area
    t = inter.pivot_table(index="pid", columns="classe", values="a", aggfunc="sum", fill_value=0)
    t = pp.set_index("pid")[["surf0"]].join(t).fillna(0)
    cl = ["pp", "herbe", "arable", "gel", "autre"]
    for c in cl:
        t[f"f_{c}"] = (t[c] / t["surf0"]).clip(0, 1) if c in t else 0.0
    t["f_absent"] = (1 - t[[f"f_{c}" for c in cl]].sum(axis=1)).clip(0, 1)
    t["devenir"] = t[[f"f_{c}" for c in cl + ["absent"]]].idxmax(axis=1).str[2:]
    t["y"] = (t["f_arable"] > 0.5).astype(int)             # conversion présumée : plus de la moitié déclarée en terre arable
    return t[["surf0", "devenir", "y"]].reset_index()

def gains(D, a0, a1):
    """Prairies permanentes de a1 dont plus de la moitié était en terre arable en a0."""
    g1 = D[a1]
    pp1 = g1.loc[(g1["classe"] == "pp") & g1["dans_dep"], ["pid", "geometry"]].copy()
    pp1["s"] = pp1.area
    ar0 = D[a0].loc[D[a0]["classe"] == "arable", ["geometry"]]
    inter = gpd.overlay(pp1[["pid", "geometry"]], ar0, how="intersection", keep_geom_type=True)
    inter["a"] = inter.area
    f = inter.groupby("pid")["a"].sum() / pp1.set_index("pid")["s"]
    return pp1[pp1["pid"].isin(f[f > 0.5].index)]

def points(geoms):
    return gpd.GeoDataFrame(geometry=gpd.GeoSeries(geoms, crs=2154).representative_point().values, crs=2154)

def au_point(pts, g, cols):
    """Attributs de la parcelle de g située sous chaque point (première trouvée)."""
    j = gpd.sjoin(pts, g[cols + ["geometry"]], how="left", predicate="within")
    return j[~j.index.duplicated()].reindex(pts.index)[cols]

def stats_maille(g, taille):
    p = g.representative_point()
    df = pd.DataFrame({"cx": (p.x // taille).astype(int), "cy": (p.y // taille).astype(int),
                       "ha": g.area / 1e4, "classe": g["classe"]})
    s = df.pivot_table(index=["cx", "cy"], columns="classe", values="ha", aggfunc="sum", fill_value=0)
    tot = s.sum(axis=1)
    return pd.DataFrame({"part_arable": s.get("arable", 0) / tot, "part_pp": s.get("pp", 0) / tot})

def commune_de(geoms, dep):
    c = COMMUNES[COMMUNES.dep == dep][["code", "nom", "codeEpci", "geometry"]]
    return au_point(points(geoms), c, ["code", "nom", "codeEpci"])

def variables(D, LAB, a0, a1=None, dep=None):
    """Variables connues l'année a0 pour chaque prairie permanente du département (aucune fuite du futur)."""
    g = D[a0]
    pp = g[(g["classe"] == "pp") & g["dans_dep"]].reset_index(drop=True)
    p = points(pp.geometry.values)
    X = pd.DataFrame({"pid": pp["pid"].values})
    X["surface_ha"] = pp.area.values / 1e4
    X["compacite"] = (4 * np.pi * pp.area / pp.length ** 2).values
    X["code_cultu"] = pp["code_cultu"].astype(str).values
    h = pd.DataFrame({a: au_point(p, D[a], ["classe"])["classe"].fillna("absent").values
                      for a in ANNEES if a < a0})
    cultivee = h.isin(["arable", "gel"])
    X["n_annees_cultivee"] = cultivee.sum(axis=1).values
    X["deja_cultivee"] = cultivee.any(axis=1).astype(int).values
    cols_pp = [a for a in h.columns if a in ANNEES_PP]
    X["deja_pt"] = (h[cols_pp] == "herbe").any(axis=1).astype(int).values if cols_pp else 0
    toujours_pp = (h[cols_pp] == "pp").all(axis=1) if cols_pp else pd.Series(True, index=h.index)
    X["ancienne"] = (~cultivee.any(axis=1) & toujours_pp).astype(int).values   # jamais cultivée depuis 2015
    X["cx1"] = (p.geometry.x // 1000).astype(int).values
    X["cy1"] = (p.geometry.y // 1000).astype(int).values
    for taille, nom in [(1000, "1km"), (5000, "5km")]:
        idx = pd.MultiIndex.from_arrays([(p.geometry.x // taille).astype(int).values,
                                         (p.geometry.y // taille).astype(int).values])
        s = stats_maille(g, taille).reindex(idx)
        X[f"part_arable_{nom}"], X[f"part_pp_{nom}"] = s["part_arable"].values, s["part_pp"].values
        prev = [a for a in ANNEES if a < a0]
        if prev:
            s0 = stats_maille(D[prev[-1]], taille).reindex(idx)
            X[f"delta_arable_{nom}"] = (s["part_arable"].values - s0["part_arable"].values)
        tp = next((t for t in TRANSITIONS if t[1] == a0), None)
        if tp:
            d = LAB[tp].merge(D[tp[0]][["pid", "geometry"]], on="pid")
            q = gpd.GeoSeries(d["geometry"], crs=2154).representative_point()
            rr = pd.DataFrame({"cx": (q.x // taille).astype(int), "cy": (q.y // taille).astype(int),
                               "y": d["y"].values}).groupby(["cx", "cy"])["y"].mean()
            X[f"taux_ret_prec_{nom}"] = rr.reindex(idx).values
    X["commune"] = commune_de(pp.geometry.values, dep)["code"].values
    if a1 is not None:
        X = X.merge(LAB[(a0, a1)][["pid", "y", "devenir"]], on="pid", how="left")
        X["transition"] = f"{a0}->{a1}"
    X["a0"] = a0
    return X

def etiquette_2ans(D, LAB, a0):
    """Horizon 2 ans : prairie de a0 encore prairie en a0+1, puis convertie entre a0+1 et a0+2."""
    if (a0 + 1, a0 + 2) not in LAB:
        return None
    g = D[a0]
    pp = g[(g["classe"] == "pp") & g["dans_dep"]].reset_index(drop=True)
    suiv = D[a0 + 1][["pid", "classe", "geometry"]].rename(columns={"pid": "pid1", "classe": "classe1"})
    a = au_point(points(pp.geometry.values), suiv, ["pid1", "classe1"])
    a["pid"] = pp["pid"].values
    a = a[a["classe1"] == "pp"]
    lab = LAB[(a0 + 1, a0 + 2)][["pid", "y"]].rename(columns={"pid": "pid1", "y": "y2"})
    return a.merge(lab, on="pid1", how="left").fillna({"y2": 0})[["pid", "y2"]]

In [ ]:
# 6. Par département : conversions présumées 2023→24 et 2024→25 (colonnes *_retournees : nom historique), étiquettes de vérification, variables pour la carte 2027
for dep in DEPS:
    fin = SORTIES / f"v4_{dep}.parquet"
    if fin.exists():
        print(f"{dep} : déjà calculé")
        continue
    t0 = time.time()
    D = {}
    for a in ANNEES:
        D[a] = charger(dep, a)
        D[a]["pid"] = np.arange(len(D[a]))           # même numérotation que la v3 (même cache)
        D[a]["classe"] = D[a]["code_group"].apply(classe)
    LAB, BILAN, OBS = {}, [], []
    for a0, a1 in [(2021, 2022), (2022, 2023), (2023, 2024), (2024, 2025)]:   # 2021-22 : variables de voisinage
        LAB[(a0, a1)] = lab = devenir(D, a0, a1)
        if a0 >= 2023:
            gsel = gains(D, a0, a1)
            BILAN.append({"dep": dep, "transition": f"{a0}->{a1}", "n_pp": len(lab), "ha_pp": round(lab.surf0.sum() / 1e4),
                          "n_retournees": int(lab.y.sum()), "ha_retournees": round(lab.loc[lab.y == 1, "surf0"].sum() / 1e4),
                          "n_sorties_rpg": int((lab.devenir == "absent").sum()),
                          "n_gains": len(gsel), "ha_gains": round(gsel["s"].sum() / 1e4)})
            ret = lab[lab.y == 1].merge(D[a0][["pid", "geometry"]], on="pid")
            OBS.append(pd.concat([commune_de(ret["geometry"].values, dep).assign(ha=ret["surf0"].values / 1e4, type="ha_retournes"),
                                  commune_de(gsel["geometry"].values, dep).assign(ha=gsel["s"].values / 1e4, type="ha_crees")])
                       .assign(dep=dep, transition=f"{a0}->{a1}"))
    # (a) Vérification : prairies 2023 encore prairies en 2024, converties entre 2024 et 2025
    verif = etiquette_2ans(D, LAB, 2023).rename(columns={"y2": "converti_2025"})
    verif.assign(dep=dep).to_parquet(SORTIES / f"verif2025_{dep}.parquet")
    # (b) Nouvelles étiquettes 2 ans pour l'entraînement : a0 = 2022 (conv. 2023→24) et a0 = 2023 (conv. 2024→25)
    nouv = []
    for a0 in [2022, 2023]:
        X = variables(D, LAB, a0, a0 + 1, dep)
        X = X.merge(etiquette_2ans(D, LAB, a0), on="pid", how="left")
        X.loc[X["y"] == 1, "y2"] = np.nan
        nouv.append(X)
    pd.concat(nouv, ignore_index=True).assign(dep=dep).to_parquet(SORTIES / f"v4_tmp_{dep}.parquet")
    # (c) Variables 2025 pour la carte de risque 2027
    variables(D, LAB, 2025, None, dep).assign(dep=dep).to_parquet(SORTIES / f"score_{dep}_2025.parquet")
    pd.DataFrame(BILAN).to_csv(SORTIES / f"bilan_v4_{dep}.csv", index=False)
    pd.concat(OBS).groupby(["dep", "code", "nom", "codeEpci", "transition", "type"], dropna=False)["ha"].sum() \
      .unstack("type", fill_value=0).round(1).reset_index().to_csv(SORTIES / f"observatoire_v4_{dep}.csv", index=False)
    (SORTIES / f"v4_tmp_{dep}.parquet").rename(fin)
    print(f"{dep} terminé ({time.time() - t0:.0f} s)")
    del D, LAB; gc.collect()

B4 = pd.concat([pd.read_csv(SORTIES / f"bilan_v4_{d}.csv", dtype={"dep": str}) for d in DEPS])
BILAN_TOUT = pd.concat([pd.read_csv(SORTIES / "bilan_transitions_normandie.csv", dtype={"dep": str}), B4])
BILAN_TOUT.to_csv(SORTIES / "bilan_transitions_2019_2025.csv", index=False)
pd.concat([pd.read_csv(SORTIES / "observatoire_communes.csv", dtype={"dep": str, "code": str, "codeEpci": str})] +
          [pd.read_csv(SORTIES / f"observatoire_v4_{d}.csv", dtype={"dep": str, "code": str, "codeEpci": str}) for d in DEPS]) \
  .to_csv(SORTIES / "observatoire_communes_2019_2025.csv", index=False)
print(BILAN_TOUT.groupby("transition")[["n_pp", "ha_pp", "n_retournees", "ha_retournees", "n_gains", "ha_gains",
                                       "n_sorties_rpg"]].sum().to_string())

In [ ]:
# 7. Outils d'évaluation (lift top 10 % avec intervalle de confiance) et modèle de la v3
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

TAB = pd.concat([pd.read_parquet(SORTIES / f"variables_{d}.parquet") for d in DEPS], ignore_index=True)
CODES = list(TAB["code_cultu"].value_counts().index[:8])
def codes_onehot(df):
    for c in CODES:
        df[f"code_{c}"] = (df["code_cultu"] == c).astype(int)
    return df
TAB = codes_onehot(TAB)
NON_VARS = {"pid", "y", "y2", "devenir", "transition", "code_cultu", "dep", "a0", "cx1", "cy1", "commune"}
VARS = [c for c in TAB.columns if c not in NON_VARS]
print("Variables :", VARS)
print(TAB.groupby(["dep", "transition"])[["y", "y2", "ancienne"]].mean().round(4).to_string())

RNG = np.random.default_rng(0)
def lift10(y, s):
    k = max(1, int(0.1 * len(y)))
    return y[np.argsort(-s)[:k]].mean() / y.mean() if y.mean() > 0 else np.nan

def evaluer(y, s, nboot=200):
    y, s = np.asarray(y, float), np.nan_to_num(np.asarray(s, float))
    if y.sum() == 0:
        return dict(AUC_PR=np.nan, lift=np.nan, lift_bas=np.nan, lift_haut=np.nan)
    boots = []
    for _ in range(nboot):
        i = RNG.integers(0, len(y), len(y))
        boots.append(lift10(y[i], s[i]))
    return dict(AUC_PR=round(average_precision_score(y, s), 4), lift=round(lift10(y, s), 2),
                lift_bas=round(np.nanpercentile(boots, 2.5), 2), lift_haut=round(np.nanpercentile(boots, 97.5), 2))

def modele():
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)

def ajuster_predire(tr, te, cible):
    """Entraîne sur les variables renseignées dans l'entraînement (une colonne vide fait planter le modèle)."""
    cols = [v for v in VARS if tr[v].notna().any()]
    m = modele().fit(tr[cols], tr[cible])
    return m.predict_proba(te[cols])[:, 1], m, cols


In [ ]:
# 8. VÉRIFICATION : le risque 2025 calculé comme en v3 (données 2023 seulement) face aux conversions observées en 2025
base_v3 = TAB[TAB["y2"].notna()]                    # exactement l'entraînement de la v3 (a0 = 2019-2021)
SC = codes_onehot(pd.concat([pd.read_parquet(SORTIES / f"score_{d}_2023.parquet") for d in DEPS], ignore_index=True))
for v in VARS:
    if v not in SC:
        SC[v] = np.nan
SC["risque"] = ajuster_predire(base_v3, SC, "y2")[0]
V = SC.merge(pd.concat([pd.read_parquet(SORTIES / f"verif2025_{d}.parquet") for d in DEPS]), on=["dep", "pid"])
print(f"Prairies 2023 encore prairies en 2024 : {len(V):,} ; converties en 2025 : {int(V.converti_2025.sum()):,}")
lignes = [{"zone": "Normandie", "n": len(V), "conversions": int(V.converti_2025.sum()),
           **evaluer(V.converti_2025, V.risque)}]
for d, g in V.groupby("dep"):
    lignes.append({"zone": d, "n": len(g), "conversions": int(g.converti_2025.sum()), **evaluer(g.converti_2025, g.risque)})
anc = V[V.ancienne == 1]
lignes.append({"zone": "Normandie, prairies anciennes", "n": len(anc), "conversions": int(anc.converti_2025.sum()),
               **evaluer(anc.converti_2025, anc.risque)})
VERIF = pd.DataFrame(lignes)
print(VERIF.to_string(index=False))
VERIF.to_csv(SORTIES / "verification_prediction_2025.csv", index=False)

# Part des conversions 2025 qui étaient dans les 10 % jugés les plus à risque (rappel)
top = V.risque >= V.risque.quantile(0.9)
print(f"\nPart des conversions 2025 situées dans les 10 % les plus à risque : "
      f"{V.loc[top, 'converti_2025'].sum() / V.converti_2025.sum():.0%}")
# À l'échelle des communes : corrélation de rang prédit / observé
com = V.groupby(["dep", "commune"]).agg(risque_prevu_ha=("surface_ha", lambda s: (s * V.loc[s.index, "risque"]).sum()),
                                         converti_ha=("surface_ha", lambda s: s[V.loc[s.index, "converti_2025"] == 1].sum()))
print("Corrélation de rang (Spearman) entre hectares à risque prévus et hectares convertis, par commune :",
      round(com["risque_prevu_ha"].corr(com["converti_ha"], method="spearman"), 3))
CRIT_PRED = bool(VERIF.loc[VERIF.zone == "Normandie", "lift"].iloc[0] >= 2)
print("\nVÉRIFICATION 2025 :", "RÉUSSIE (lift ≥ 2)" if CRIT_PRED else "ÉCHEC (lift < 2)",
      "| Mesure 2025 :", "OK" if CRIT_2025 else "ÉCHEC")